# Open Government Toolcalling Agent

## Konzept und Funktionsweise

Dieses Notebook implementiert einen KI-Agenten, der komplexe Suchanfragen auf `data.gv.at` in natürlicher Sprache autonom in spezifische Teilabfragen (Sub-Queries) zerlegt. Jede Sub-Query wird über die hybride Suche ausgeführt. 

Der Agent evaluiert die zurückgegebenen Metadaten selbstständig auf ihre Relevanz. Sind die Ergebnisse unzureichend, passt er die Suchparameter an und führt alternative Abfragen durch (limitiert auf maximal $n$ Iterationen, um Endlosschleifen zu vermeiden). Zum Abschluss liefert der Agent eine strukturierte Liste relevanter Datensätze inklusive URLs und einer kurzen Metadaten-Zusammenfassung.

## System-Logik

Der System-Prompt zwingt das Modell in folgenden iterativen Problemlösungsprozess:

1. **Analyse (Decomposition):** Zerlegung komplexer Anfragen in granulare Sub-Queries 
2. **Recherche (Tool Use):** Iterativer Aufruf des Tools `search_government_data` zur Abarbeitung der Teilabfragen.
3. **Relaxation (Fallback):** Automatische Verallgemeinerung der Suchparameter, falls eine spezifische query keine oder unzureichende Treffer liefert.
4. **Formatierung:** Strukturierte Ausgabe der validierten Treffer mit anklickbaren HTML-Links (`<a href="URL">Titel</a>`).
5. **Vollständigkeits-Toleranz:** Ausgabe von Teilantworten, falls die verfügbaren Daten nicht alle Aspekte der ursprünglichen Nutzerfrage abdecken.

## Initialisation

Before you run this script, make sure to install the following Python libraries. This project requires **Python 3.12**.

```bash
pip install google-genai==1.74.0 requests==2.33.1 python-dotenv==1.2.2
```

In [3]:
import json
import os
import google.genai as genai
from google.genai import types
import requests
from dotenv import load_dotenv

In [9]:
# config
SEARCH_URL = "https://www.data.gv.at/api/hub/search/search"

load_dotenv() 

api_key = os.environ.get("GEMINI_API_KEY") # get GEMINI_API_KEY at https://ai.google.dev/gemini-api/docs/api-key
if not api_key:
    raise ValueError("GEMINI_API_KEY setzen")

client = genai.Client()

In [10]:
# tool initialisation

def execute_search_tool(q: str, limit: int = 5) -> dict:
    print(f"\n[System] Tool-Aufruf: '{q}'")
    url = "https://www.data.gv.at/api/hub/search/search"
    params = {"q": q, "limit": limit}
    
    try:
        response = requests.get(SEARCH_URL, params=params)
        response.raise_for_status()
        data = response.json()
        results = data.get("result", {}).get("results", [])
        
        if not results:
            return {"status": "no_results", "message": f"Keine Treffer fuer '{q}'."}
        
        extracted_info = []
        for item in results:

            title_obj = item.get("title", {})
            desc_obj = item.get("description", {})
            title = title_obj.get("de", str(title_obj)) if isinstance(title_obj, dict) else str(title_obj)
            desc = desc_obj.get("de", str(desc_obj)) if isinstance(desc_obj, dict) else str(desc_obj)
            
            ds_id = item.get("id", "")
            portal_url = f"https://www.data.gv.at/katalog/dataset/{ds_id}"
            
            extracted_info.append({
                "title": title,
                "description": desc[:200] + "...",
                "url": portal_url
            })
            
        return {"status": "success", "data": extracted_info}
    except Exception as e:
        return {"status": "error", "message": str(e)}

search_tool_definition = {
    "name": "search_government_data",
    "description": "Sucht Datensaetze auf data.gv.at.",
    "parameters": {
        "type": "object",
        "properties": {
            "q": {"type": "string", "description": "Suchbegriff."},
            "limit": {"type": "integer", "description": "Anzahl Ergebnisse."}
        },
        "required": ["q"],
    },
}

In [11]:
def run_agentic_rag(user_prompt: str):
    client = genai.Client()
    tools = types.Tool(function_declarations=[search_tool_definition])
    
    system_instruction = """Du bist ein Experte fuer offene Daten in Oesterreich.
    
    ARBEITSWEISE:
    1. ANALYSE: Zerlege komplexe Anfragen in Sub-Queries. Wenn jemand nach 'Wahldaten Tirol Schwaz' sucht, suche separat nach 'Landtagswahl Tirol Schwaz', 'Nationalratswahl Tirol Schwaz' etc.
    2. RECHERCHE: Nutze das Tool 'search_government_data' mehrfach, um diese Sub-Queries abzuarbeiten.
    3. RELAXATION: Wenn eine spezifische Suche (z.B. mit Bezirk) nichts liefert, suche allgemeiner (nur Bundesland oder nur Thema).
    4. ANTWORT: Erstelle eine Liste der gefundenen Datensaetze. Nutze fuer die Links das Format: <a href="URL">Titel</a>.
    5. VOLLSTÄNDIGKEIT: Wenn du Daten gefunden hast, gib sie aus, auch wenn sie nicht alle Aspekte der Frage abdecken."""

    messages = [types.Content(role="user", parts=[types.Part.from_text(text=user_prompt)])]
    config = types.GenerateContentConfig(tools=[tools], system_instruction=system_instruction, temperature=0.0)

    max_iterations = 6
    for i in range(max_iterations):
        response = client.models.generate_content(
            model="gemini-3-flash-preview",
            contents=messages,
            config=config,
        )

        # Fallback, falls Modell eine Antwort ohne Toolcall returned
        if not (response.candidates and response.candidates[0].content.parts and response.candidates[0].content.parts[0].function_call):
            print("\n--- FINALE ANTWORT ---")
            print(response.text)
            return

        # Tool-Calls abarbeiten
        messages.append(response.candidates[0].content)
        for part in response.candidates[0].content.parts:
            if part.function_call:
                call = part.function_call
                result = execute_search_tool(**dict(call.args))
                
                messages.append(types.Content(
                    role="user",
                    parts=[types.Part.from_function_response(name=call.name, response=result)]
                ))

    # 3. Fallback nach max_iterations
    print("\n[System] Maximale Iterationen erreicht. Erzwinge Zusammenfassung...")
    final_prompt = "Fasse jetzt alle bisher gefundenen Ergebnisse zusammen. Wenn fuer bestimmte Teilbereiche nichts gefunden wurde, erwaehne das kurz. Gib alle Links als HTML-Links aus."
    messages.append(types.Content(role="user", parts=[types.Part.from_text(text=final_prompt)]))
    
    final_res = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=messages,
        config=types.GenerateContentConfig(system_instruction=system_instruction)
    )
    print("\n--- FINALE ANTWORT (FALLBACK) ---")
    print(final_res.text)

## Proof of Concept: Agentic Search vs. Klassische Suche

**Szenario:** Eine Nutzerin sucht nach *"Vergleich der Gesundheitsausgaben und des Rauchverhaltens in Wien"*.
*   **Klassische Suche (Status Quo):** Eine Eingabe dieser Phrase führt zu keinen oder völlig irrelevanten Treffern. Die Nutzerin müsste manuell unzählige Suchbegriffe (Trial-and-Error) ausprobieren, die Ergebnisse händisch filtern und selbst abgleicht.

    Top 5 Treffer der klassischen Suche (Elastic Search)

1. Vergleichende Fischbestandserhebungen am Johnsbach und der Enns im Rahmen des EU LIFE-Projektes
2. Bericht - Laichhilfen Wienfluss 2023 Wien
3. Volksbegehren 2018 Vergleich in Prozent der Gemeinde Engerwitzdorf
4. Verkehrsflächen des Bundes und des Landes in der Gemeinde Engerwitzdorf
5. Finanzielle Ausstattung und Leistungsfähigkeit der Stadt Graz im Vergleich


*   **Unser Toolcalling-Agent in Aktion:** 
    1. **Autonome Recherche:** Der Agent erkennt den komplexen Intent und führt im Hintergrund selbstständig 15 iterative Sub-Queries aus (z. B. nach *'Gesundheitsausgaben Wien'*, *'Rauchverhalten Wien'*, *'ATHIS'* oder *'Rechnungsabschluss Wien'*), wobei er die Parameter schrittweise anpasst.
    2. **Synthese & Ergebnis:** Der Agent versteht die Zusammenhänge und liefert eine fertige, kuratierte Liste: Er identifiziert das *Statistische Jahrbuch der Stadt Wien* als zentrale Quelle für beide Parameter und ergänzt die Antwort zielgenau um spezifische CSV-Datensätze zu laufenden Gesundheitsausgaben und verhaltensbezogenen Gesundheitsstudien (siehe Toolaufruf unten)

In [12]:
run_agentic_rag("Vergleich der Gesundheitsausgaben und des Rauchverhaltens in Wien")


[System] Tool-Aufruf: 'Gesundheitsausgaben Wien'

[System] Tool-Aufruf: 'Rauchverhalten Wien'

[System] Tool-Aufruf: 'Gesundheitsbefragung Wien'

[System] Tool-Aufruf: 'ATHIS'

[System] Tool-Aufruf: 'Rauchen Bundesländer'

[System] Tool-Aufruf: 'Gesundheitsbericht Wien'

[System] Tool-Aufruf: 'Rechnungsabschluss Wien Gesundheit'

[System] Tool-Aufruf: 'Gesundheitsbefragung'

[System] Tool-Aufruf: 'Rechnungsabschluss Wien'

[System] Tool-Aufruf: 'Gesundheitsverhalten'

[System] Tool-Aufruf: 'Statistisches Jahrbuch Wien'

[System] Tool-Aufruf: 'Wiener Gesundheitsbefragung'

[System] Tool-Aufruf: 'Gesundheitsausgaben Österreich Bundesländer'

[System] Tool-Aufruf: 'MA 24 Wien'

[System] Tool-Aufruf: 'MA 15 Wien'

--- FINALE ANTWORT ---
Für einen Vergleich der Gesundheitsausgaben und des Rauchverhaltens in Wien stehen auf data.gv.at verschiedene Datensätze zur Verfügung. Die umfassendste Quelle für beide Themenbereiche ist das Statistische Jahrbuch der Stadt Wien, das detaillierte Zeitrei

## Limitationen & Systemgrenzen

Obwohl der agentenbasierte Ansatz die Suchqualität massiv erhöht, bringt die aktuelle Architektur zwei zentrale Herausforderungen mit sich:

1. **Abhängigkeit von LLM-Kapazitäten (Tool-Calling & Context Window):** 
   Das System erfordert ein leistungsstarkes Large Language Model, das natives Tool-Calling zuverlässig unterstützt und über ein ausreichend großes Context Window verfügt. Nur so kann der Agent mehrere komplexe Metadaten-Ergebnisse aus verschiedenen Suchiterationen gleichzeitig im Speicher halten, vergleichen und synthetisieren. Dies schränkt die Auswahl der nutzbaren (insbesondere kleinerer, lokaler) Modelle ein und kann zu höheren API-Kosten führen.

2. **Latenz & erhöhter Traffic (Infrastruktur-Auslastung):** 
   Im Gegensatz zu einer klassischen, singulären Suchanfrage (Elastic Search) führt der Agent pro User-Query iterativ mehrere Datenbankabfragen im Hintergrund durch. Dies führt naturgemäß zu einer höheren Antwortlatenz für den Endnutzer. Gleichzeitig vervielfacht dieser Ansatz die Netzwerkanfragen (Traffic) auf die Infrastruktur von `data.gv.at`, was bei einer Skalierung auf tausende Nutzer entsprechende Caching-Strategien (siehe Future Work) zwingend erforderlich macht.